<div dir="rtl" align="right">

# مقیاس‌بندی و کدگذاری ویژگی‌ها

درسی از مدرسه پایتون و هوش مصنوعی، دکتر بهروز طاهری

## چرا مقیاس‌بندی؟
فرض کن دو ویژگی داری: سن (۲۰ تا ۸۰) و حقوق (۱۰۰۰ تا ۵۰۰۰). در فاصله اقلیدسی، اختلاف ۱۰۰۰ تومانی حقوق ۱۰۰۰ برابر اختلاف یک سال سن حساب می‌شود؛ یعنی **سن عملاً نادیده گرفته می‌شود**.

[[lab:scaling]]

## کدام مدل‌ها نیاز دارند؟
| نیاز دارند | نیاز ندارند |
|---|---|
| KNN، K-Means، DBSCAN (فاصله) | درخت تصمیم |
| SVM، SVR | جنگل تصادفی |
| رگرسیون با منظم‌سازی (Ridge، Lasso) | گرادیان بوستینگ |
| شبکه عصبی و هر مدل با گرادیان کاهشی | بیز ساده |
| PCA | |

## روش‌ها
| روش | فرمول | نتیجه | مناسب برای |
|---|---|---|---|
| `StandardScaler` | $z=(x-\mu)/\sigma$ | میانگین ۰، انحراف معیار ۱ | پیش‌فرض خوب |
| `MinMaxScaler` | $(x-\min)/(\max-\min)$ | بین ۰ و ۱ | تصویر، شبکه عصبی |
| `RobustScaler` | $(x-\text{median})/IQR$ | مقاوم به داده پرت | داده با پرت |
| تبدیل لگاریتم | $\log(1+x)$ | چولگی کمتر | درآمد، قیمت، شمارش |

## مثال عددی
سن‌ها: 20، 30، 40، 50، 60. میانگین 40 و انحراف معیار 14.14:

| سن | Standard | MinMax |
|---|---|---|
| 20 | −1.41 | 0.00 |
| 30 | −0.71 | 0.25 |
| 40 | 0.00 | 0.50 |
| 50 | 0.71 | 0.75 |
| 60 | 1.41 | 1.00 |



</div>

In [ ]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler, RobustScaler, StandardScaler

age = np.array([[20], [30], [40], [50], [60], [95]])   # 95 is an outlier
for s in [StandardScaler(), MinMaxScaler(), RobustScaler()]:
    print(f"{type(s).__name__:15s}", s.fit_transform(age).ravel().round(2))

<div dir="rtl" align="right">



## کدگذاری ویژگی‌های دسته‌ای
| روش | مثال | کی |
|---|---|---|
| One-Hot | رنگ ← سه ستون قرمز/سبز/آبی با ۰ و ۱ | دسته‌های بدون ترتیب |
| Ordinal | کم=۰، متوسط=۱، زیاد=۲ | دسته‌های **ترتیب‌دار** |
| Target encoding | میانگین هدف برای هر دسته | تعداد دسته خیلی زیاد (با احتیاط) |

> دادن عدد ۰، ۱، ۲ به «تهران، شیراز، تبریز» یعنی به مدل بگویی شیراز بین تهران و تبریز است! برای دسته‌های بی‌ترتیب One-Hot لازم است.

## نشت داده
مقیاس‌کننده را **فقط روی داده آموزش** fit کن؛ وگرنه اطلاعات داده آزمون (میانگین و انحراف معیارش) به آموزش نشت می‌کند. `Pipeline` و `ColumnTransformer` این را خودکار درست انجام می‌دهند:



</div>

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler

df = pd.DataFrame({
    "age": [25, 32, 47, 51, 23, 36, 60, 29, 41, 55],
    "salary": [1800, 2500, 4100, 3900, 1500, 3000, 4500, 2100, 3300, 4200],
    "city": ["Tehran", "Shiraz", "Tehran", "Tabriz", "Shiraz", "Tabriz", "Tehran", "Shiraz", "Tabriz", "Tehran"],
    "level": ["low", "medium", "high", "high", "low", "medium", "high", "low", "medium", "high"],
    "bought": [0, 0, 1, 1, 0, 1, 1, 0, 1, 1],
})
X, y = df.drop(columns="bought"), df["bought"]
prep = ColumnTransformer([
    ("num", StandardScaler(), ["age", "salary"]),
    ("city", OneHotEncoder(handle_unknown="ignore"), ["city"]),
    ("level", OrdinalEncoder(categories=[["low", "medium", "high"]]), ["level"]),
])
model = Pipeline([("prep", prep), ("clf", LogisticRegression())])
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
model.fit(X_tr, y_tr)                 # the scaler only sees the training rows
print(prep.get_feature_names_out())
print("Test accuracy:", model.score(X_te, y_te))